# Unsupervised Learning

## Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio; pio.renderers.default = "notebook"
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Dataset Load

In [2]:
all_players = pd.read_csv("../Data/Clean/All_Players.csv")
all_players_2024 = all_players[all_players["season"] == 2024].copy()

Note: Here, only the 2024 season is analyzed!!!

## Standardization

Before applying $K$-means clustering or PCA, each feature is standardized. The features in this dataset are measured in very different units (e.g., miles per hour, feet, counts, and proportions), so their scales differ by orders of magnitude. Since $K$-means relies on Euclidean distances and PCA relies on variances and covariances, features with large numerical ranges, such as `maximum_distance_hit`, would otherwise dominate the results simply because of their units, not because they are more important.

For the $i\text{th}$ feature, with sample mean $\overline{x}_i$ and sample standard deviation $s_i$, each observation $x_{ni}$ is replaced by its $Z$-score:

$$
\tilde{x}_{ni}=\frac{x_{ni}-\overline{x}_i}{s_i} \tag*{(1)}
$$

so that every standardized feature has mean $0$ and standard deviation $1$. This is exactly what `StandardScaler` does.

A useful consequence is that the covariance between two standardized features equals the correlation between the original features:

$$
\mathrm{Cov}\left(\tilde{X}_i,\,\tilde{X}_j\right)=\frac{\mathrm{Cov}(X_i,\,X_j)}{s_i\,s_j}=r_{ij} \tag*{(2)}
$$

Hence, PCA on standardized data is equivalent to PCA on the correlation matrix of the original features.

## $K$-means Clustering

The primary goal of $K$-means clustering is to minimize the inertia metric: the total within-cluster sum of squares.

Let a given dataset consist of $N$ observations $\vec{x}_1,\,\vec{x}_2,\,\ldots,\,\vec{x}_N\in\mathbb{R}^p$ where each observation has $p$ features.

If the data are partitioned into $K$ clusters $C_1,\,C_2,\,\ldots,\,C_K$ with corresponding centroids $\vec{\alpha}_1,\,\vec{\alpha}_2,\,\ldots,\,\vec{\alpha}_K\in\mathbb{R}^p$, then the inertia is 

$$
\iota=\sum_{k=1}^K\sum_{\vec{x}_n\in C_k}\|\vec{x}_n-\vec{\alpha}_k\|^2 \tag*{(3)}
$$

The $K$-means algorithm seeks to minimize $\iota$ by iteratively updating the centroids and reassigning observations to their nearest cluster centers. Each iteration can only decrease $\iota$ (or leave it unchanged), so the algorithm is guaranteed to converge. However, it converges to a local minimum, not necessarily the global minimum, and the result depends on where the centroids are initialized. For this reason, `sklearn`'s `KMeans` uses the $K$-means + initialization, which spreads the initial centroids apart by choosing each new centroid with probability proportional to its squared distance from the nearest centroid already chosen, and it runs the algorithm from several initializations, keeping the result with the lowest inertia.

### The Silhouette Score

For a clustering algorithm, the silhouette score at a point $\vec{x}_n$ can be used to evaluate the clustering result and help determine the best choice of $K$ (a hyperparameter).

If $\vec{x}_n\in C_k$, then the silhouette score is 

$$
s_n=\frac{b_n-a_n}{\max\{a_n,\,b_n\}} \tag*{(4)}
$$

where

$$
a_n=\frac{1}{|C_k|-1}\sum_{\vec{x}_m\in C_k,\,m\neq n}d(\vec{x}_m,\,\vec{x}_n) \tag*{(5)}
$$

is the average distance from $\vec{x}_n$ to all other points in its own cluster, using the Euclidean distance $d$, and

$$
b_n=\min_{j\neq k}\left\{\frac{1}{|C_j|}\sum_{\vec{x}_{\ell}\in C_j}d(\vec{x}_{\ell},\,\vec{x}_n)\right\} \tag*{(6)}
$$

is the smallest average distance from $\vec{x}_n$ to points in another cluster.

The overall silhouette score $\varphi$ for the clustering is the mean of all pointwise scores $s_n$:

$$
\varphi=\frac{1}{N}\sum_{n=1}^Ns_n \tag*{(7)}
$$

Finally, values of $s_n$ close to $1$ indicate well-separated clusters, values near $0$ indicate overlapping clusters, and negative values suggest possible misclassifications.

Note: The Euclidean distance between two points $\vec{x}_1,\,\vec{x}_2\in\mathbb{R}^p$ can be described as the function $d\colon\mathbb{R}^p\times\mathbb{R}^p\to\mathbb{R}$ where

$$
d(\vec{x}_1,\,\vec{x}_2)=\sqrt{\sum_{i=1}^p(x_{2i}-x_{1i})^2}=\|\vec{x}_2-\vec{x}_1\| \tag*{(8)}
$$

Also, $a_n$ is undefined when a point $\vec{x}_n$ is alone in its cluster $C_k$, as $|C_k|-1$ is the denominator in the formula for $a_n$. `sklearn` handles this edge case by setting $s_n=0$. For similar reasons, the silhouette score requires $2\leq K\leq N-1$. If $K=1$, there is no other cluster, so $b_n$ is undefined. If $K=N$, every point is alone in its own cluster, so every $s_n=0$ and the score carries no information.

## Principal Component Analysis

Principal component analysis (PCA) is a dimensionality reduction technique used to visualize high-dimensional data by constructing the $2$ or $3$ most significant principal components. 

These principal components are orthogonal linear combinations of the original features and correspond to the eigenvectors of the covariance matrix associated with the largest eigenvalues, representing the directions of maximum variance in the data.

After standardizing each feature (see the note above), the covariance matrix for a $p$-dimensional feature space is given by

$$
\Psi 
=
\left(
    \begin{array}{cccc}
        \mathrm{Var}(X_1) & \mathrm{Cov}(X_1,\,X_2) & \cdots & \mathrm{Cov}(X_1,\,X_p) \\
        \mathrm{Cov}(X_2,\,X_1) & \mathrm{Var}(X_2) & \cdots & \mathrm{Cov}(X_2,\,X_p) \\
        \vdots & \vdots & \ddots & \vdots \\
        \mathrm{Cov}(X_p,\,X_1) & \mathrm{Cov}(X_p,\,X_2) & \cdots & \mathrm{Var}(X_p)
    \end{array}
\right)
\in
\mathbb{R}^{p\times p}
\tag*{(9)}
$$

Here, $X_1,\,X_2,\,\ldots,\,X_p$ denote the standardized features. Since each one has variance $1$, every diagonal entry of $\Psi$ equals $1$, and each off-diagonal entry is the correlation $r_{ij}$ between the original features $i$ and $j$. In other words, $\Psi$ is the correlation matrix.

To find the eigenvalues $\lambda_i$ of $\Psi$ for $1\leq i\leq p$, one solves the following:

$$
\det\left(\Psi-\lambda\mathcal{I}\right)=0 \tag*{(10)}
$$

$\mathcal{I}\in\mathbb{R}^{p\times p}$ is the identity matrix!

Since $\Psi$ is symmetric, its eigenvalues are real, and since $\vec{v}^\top\Psi\,\vec{v}$ is the variance of the data projected onto a unit vector $\vec{v}$, which can never be negative, its eigenvalues are also nonnegative $(\lambda_i\geq0)$. In fact, $\lambda_i$ is exactly the variance of the data along the $i\text{th}$ principal component.

Ordering the eigenvalues so that $\lambda_1\geq\lambda_2\geq\cdots\geq\lambda_p$, the first principal component corresponds to $\lambda_1$, the second to $\lambda_2$, and so on.

Then, to find the eigenvector $\vec{v}_i\in\mathbb{R}^p$ that corresponds to each eigenvalue $\lambda_i$, one solves the following system of equations for the components of $\vec{v}_i$:

$$
\Psi\,\vec{v}_i=\lambda_i\,\vec{v}_i \tag*{(11)}
$$

In the $2$-dimensional case, for some $\lambda\in\mathbb{R}$ that corresponds to $\vec{v}\in\mathbb{R}^2$, this system looks like

$$
\begin{cases}
    \psi_{11}v_1+\psi_{12}v_2=\lambda v_1 \\
    \psi_{21}v_1+\psi_{22}v_2=\lambda v_2
\end{cases}
\tag*{(12)}
$$

where $\Psi\in\mathbb{R}^{2\times2}$ and $\psi_{ij}$ are the entries of $\Psi$.

To decide how many principal components to keep, one can compute the proportion of the total variance explained by the $i\text{th}$ principal component, called its explained-variance ratio:

$$
\frac{\lambda_i}{\sum_{j=1}^p\lambda_j}=\frac{\lambda_i}{\mathrm{tr}(\Psi)} \tag*{(13)}
$$

where $\mathrm{tr}(\Psi)$ is the trace of $\Psi$, the sum of its diagonal entries. Since the features are standardized, every diagonal entry of $\Psi$ is $1$, so $\mathrm{tr}(\Psi)=p$. Keeping $2$ or $3$ components is well justified when their explained-variance ratios add up to a large share of the total. These values are available in `sklearn` as the `explained_variance_ratio_` attribute of a fitted `PCA` object.

Finally, the data are projected onto the leading eigenvectors to obtain lower-dimensional coordinates for visualization and analysis.

Note: The eigenvectors are normalized $(\|\vec{v}_i\|=1)$, as otherwise the projections aren't uniquely scaled. Even then, each eigenvector is only unique up to its sign, since $-\vec{v}_i$ is also a unit eigenvector for $\lambda_i$, so a principal component's axis may appear flipped across different software or runs.